# Monte Carlo robustness study

The strategy's one measured history is a single draw. This notebook uses
the repository's stationary block-bootstrap machinery
(`amat.validation.montecarlo`) to resample the market panel's own joint
session moves into alternative markets, run the **frozen v1 core** on
each, and place the observed result inside the resampled distribution.

**What this is not.** A forecast. The bootstrap preserves cross-sectional
structure exactly and short-horizon clustering approximately, but it
breaks dependence longer than its blocks — which is conservative for a
trend strategy, whose premium (if any) *is* long memory. Probabilities
below describe the resampled worlds only.

*Backtest research only; nothing here is investment advice or a
live-trading signal. The canonical, full-sample, record-persisting entry
point is `amat montecarlo` — this notebook is the same machinery on a
reduced window so it executes in minutes.*

In [1]:
# Locate the repository root whether this kernel started in notebooks/ or the root.
import datetime as dt
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
assert (ROOT / "configs" / "strategy" / "amat_v1.toml").exists(), ROOT

import matplotlib
matplotlib.use("Agg")  # headless-safe; inline display still works
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from amat.config.loading import load_strategy_config
from amat.data.snapshot import load_corporate_actions, load_panel

SNAPSHOT = ROOT / "data" / "snapshots" / "alpaca-sip-2026-08-11"
resolved = load_strategy_config(ROOT / "configs" / "strategy" / "amat_v1.toml")
strategy = resolved.strategy
symbols = [str(s) for s in strategy.universe.symbols]

# A reduced window keeps every notebook run affordable (minutes, not an hour).
# The CLI commands run the full sample and persist run records; this notebook
# demonstrates the same machinery through the library API.
START = dt.date(2021, 1, 4)
panel, manifest = load_panel(SNAPSHOT, symbols, start=START)
actions = load_corporate_actions(SNAPSHOT, symbols, sessions=panel.sessions)
print(f"snapshot {manifest.snapshot_id}: {len(panel)} sessions from {panel.sessions[0]}")
print(f"config {resolved.content_sha256[:16]}  strategy {strategy.name} v{strategy.version}")


Matplotlib is building the font cache; this may take a moment.


snapshot alpaca-sip-2026-08-11: 1406 sessions from 2021-01-04
config ea8cbd5ad7418a38  strategy amat-etf v1.0.0


## One resampled market

A resampled session is one whole cross-sectional row of per-symbol moves
(never a per-symbol shuffle), drawn in geometric-length blocks with mean
21 sessions. Same calendar, same starting prices; dividends live inside
the price path, so the resampled world carries no separate corporate
actions.

In [2]:
from amat.validation.montecarlo import bootstrap_panel

spy = symbols.index("SPY")
fig, ax = plt.subplots(figsize=(10, 4.5))
for seed in (1, 2, 3):
    resampled = bootstrap_panel(panel, seed=seed)
    ax.plot(panel.sessions,
            resampled.adjusted_closes[:, spy] / resampled.adjusted_closes[0, spy],
            linewidth=0.9, alpha=0.7, label=f"resampled (seed {seed})")
ax.plot(panel.sessions, panel.adjusted_closes[:, spy] / panel.adjusted_closes[0, spy],
        color="black", linewidth=1.8, label="observed SPY")
ax.set_title("SPY, observed vs three bootstrap worlds (growth of 1, adjusted)")
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)
plt.show()

/tmp/claude-1000/ipykernel_27/138362476.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The study

Each path is a full engine run — warm-up, next-open VWAP execution,
2 bps one-way costs, accounting identity — on one resampled market.
Path *i* uses seed `base + i`, so any path can be regenerated alone and
the study reproduces bit-for-bit at any worker count.

In [3]:
from decimal import Decimal
from amat.validation.montecarlo import run_monte_carlo

study = run_monte_carlo(
    strategy,
    panel,
    strategy.to_universe(),
    paths=16,                    # the CLI runs 200; 16 keeps the notebook to minutes
    seed=20260812,
    initial_cash=Decimal("100000"),
    actions_by_session=actions,  # the observed run credits real dividends/splits
    workers=4,
    progress=print,
)
print()
print(study.caveat)

observed run: 1406 sessions, 12 symbols


resampling: 16 paths, 4 worker(s), base seed 20260812


  16/16 paths complete



Resampled distribution, not a forecast: the bootstrap preserves the sample's joint session moves and short-horizon clustering but breaks dependence longer than its blocks, which is conservative for a trend strategy. Probabilities describe the resampled worlds only.


In [4]:
# Equity fan: percentile bands of the resampled paths, observed curve overlaid.
fig, ax = plt.subplots(figsize=(10, 4.5))
dates = list(study.band_dates)
ax.fill_between(dates, study.equity_bands["p05"], study.equity_bands["p95"],
                color="#c7d8ec", linewidth=0, label="resampled 5th-95th pct")
ax.fill_between(dates, study.equity_bands["p25"], study.equity_bands["p75"],
                color="#8fb4dc", linewidth=0, label="resampled 25th-75th pct")
ax.plot(dates, study.equity_bands["p50"], "--", color="#2f6db3", linewidth=1.1,
        label="resampled median")
ax.plot(dates, study.observed_equity, color="black", linewidth=1.8, label="observed")
ax.axhline(study.initial_cash, color="grey", linewidth=0.8, linestyle=":")
ax.set_title(f"Equity fan - {study.strategy}, {study.paths} bootstrap paths")
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)
plt.show()

/tmp/claude-1000/ipykernel_27/766124206.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# The distribution table the CLI's PDF renders, as a DataFrame.
rows = []
for metric, pcts in study.metric_percentiles.items():
    rows.append({
        "metric": metric,
        **{k: round(v, 4) for k, v in pcts.items()},
        "observed": round(getattr(study.observed, metric), 4),
        "obs_percentile": round(study.observed_percentile[metric], 2),
    })
table = pl.DataFrame(rows)
print(table)
print()
for name, value in study.probabilities.items():
    print(f"P[{name}] = {value:.1%} of resampled paths")

shape: (5, 8)
┌────────────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┬────────────────┐
│ metric             ┆ p05     ┆ p25     ┆ p50     ┆ p75     ┆ p95     ┆ observed ┆ obs_percentile │
│ ---                ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---            │
│ str                ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64      ┆ f64            │
╞════════════════════╪═════════╪═════════╪═════════╪═════════╪═════════╪══════════╪════════════════╡
│ total_return       ┆ 0.1235  ┆ 0.24    ┆ 0.3881  ┆ 0.4619  ┆ 0.5204  ┆ 0.3586   ┆ 0.44           │
│ annualized_return  ┆ 0.0257  ┆ 0.0481  ┆ 0.0742  ┆ 0.0865  ┆ 0.0958  ┆ 0.0692   ┆ 0.44           │
│ annualized_volatil ┆ 0.0649  ┆ 0.0689  ┆ 0.072   ┆ 0.0725  ┆ 0.0748  ┆ 0.0696   ┆ 0.31           │
│ ity                ┆         ┆         ┆         ┆         ┆         ┆          ┆                │
│ sharpe             ┆ 0.3637  ┆ 0.7054  ┆ 1.0291  ┆ 1.1843  ┆ 1.3288  ┆ 0.99

## Reading the result honestly

- An observed metric sitting **above** most resampled paths means the
  real ordering of history (long trends the blocks destroy) — or luck —
  carried it; the bootstrap cannot distinguish those two.
- The exceedance probabilities are *within-resampling* frequencies, not
  probabilities about the future.
- The full-sample, 200-path study with a persisted run record and a PDF:

```bash
uv run amat montecarlo configs/strategy/amat_v1.toml \
  --snapshot data/snapshots/alpaca-sip-2026-08-11 \
  --paths 200 --pdf outputs/reports/amat_v1_montecarlo.pdf
```